# Onboarding Next-Field Transformer -- Final Edition

This is the complete, final notebook: data -> model -> training -> evaluation
-> generalization proof -> edge-case testing -> defensive input handling.
Self-contained -- upload `onboarding_dataset.csv` and run top to bottom.

## What this model does
Predicts the next field a banking/telecom onboarding session will ask for,
given the fields entered so far, the industry, session context
(individual/business, or telecom service type), and the validation status
of the most recently completed field.

## Why this architecture
A standard classifier's output layer is `logits = session_vector @ W + b`,
with one independently-learned vector per class, fixed at training time --
no way to add a class later. This model's output layer instead compares
the session vector against **field description vectors**, produced by a
shared, trainable text encoder. The 33 known fields' vectors come from
that encoder; a brand-new field, described in the same style, gets a real
vector from the same trained encoder with **zero retraining**. Section 9
proves this works on the real trained model, not just in theory.

## Leakage and correctness controls
- `real_time_guidance` (and `next_onboarding_field`, the label itself) are
  never fed as model inputs -- `real_time_guidance` literally names the
  next field in its text about a third of the time.
- `validation_status` used as a feature is read only from the
  **already-completed** step, never the target step.
- Train/validation/test are split by **session** (the CSV's `split`
  column), so no session's rows cross splits.
- Sequences are left-padded; `x[:, -1, :]` pooling is only valid because of
  that, and it's asserted against the real data before being relied on.

## 1. Setup

In [ ]:
import json
import time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

DATASET_PATH = "onboarding_dataset.csv"

## 2. Load the dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
df = df.sort_values(["session_id", "event_timestamp"])
print("Rows:", len(df), " Sessions:", df["session_id"].nunique())
print(df["split"].value_counts().to_dict())
print("Unique fields (classes):", df["onboarding_field"].nunique(), "(should be 33)")

## 3. Preprocess

Sequences are left-padded; the assertion confirms the final position is
always the most recent real token, which is what makes `x[:, -1, :]`
pooling valid.

In [ ]:
field_vocab = sorted(df["onboarding_field"].unique().tolist())
field_to_id = {f: i + 1 for i, f in enumerate(field_vocab)}  # 0 = padding
NUM_CLASSES = len(field_vocab)
MAX_LEN = int(df.groupby("session_id").size().max())

industry_to_id = {v: i for i, v in enumerate(sorted(df["industry"].unique()))}
context_to_id = {v: i for i, v in enumerate(sorted(df["context_value"].unique()))}
validation_to_id = {"NONE": 0, "passed": 1, "failed": 2}
print("MAX_LEN:", MAX_LEN, " NUM_CLASSES:", NUM_CLASSES)


def build_split(split_name):
    sub = df[df["split"] == split_name]
    xs, masks, ys, industries, contexts, validations = [], [], [], [], [], []
    for _, g in sub.groupby("session_id"):
        g = g.reset_index(drop=True)
        field_ids = [field_to_id[f] for f in g["onboarding_field"]]
        industry_id = industry_to_id[g["industry"].iloc[0]]
        context_id = context_to_id[g["context_value"].iloc[0]]
        for i in range(len(field_ids) - 1):
            prefix = field_ids[:i + 1]
            xs.append([0] * (MAX_LEN - len(prefix)) + prefix)
            masks.append([0] * (MAX_LEN - len(prefix)) + [1] * len(prefix))
            ys.append(field_ids[i + 1] - 1)
            industries.append(industry_id)
            contexts.append(context_id)
            validations.append(validation_to_id.get(g["validation_status"].iloc[i], 0))
    return (np.array(xs, np.int32), np.array(masks, bool), np.array(ys, np.int32),
            np.array(industries, np.int32), np.array(contexts, np.int32),
            np.array(validations, np.int32))


(train_x, train_mask, train_y, train_ind, train_ctx, train_val) = build_split("train")
(val_x, val_mask, val_y, val_ind, val_ctx, val_val) = build_split("validation")
(test_x, test_mask, test_y, test_ind, test_ctx, test_val) = build_split("test")

for name, (x, mask) in [("train", (train_x, train_mask)), ("validation", (val_x, val_mask)),
                         ("test", (test_x, test_mask))]:
    assert mask[:, -1].all(), f"{name}: left-padding assumption violated"
    print(name, x.shape)
print("Left-padding verified.")

## 4. Ceiling check (before training)

In [ ]:
key_targets = defaultdict(Counter)
for seq, mask, ctx, v, target in zip(train_x, train_mask, train_ctx, train_val, train_y):
    key_targets[(tuple(seq[mask].tolist()), int(ctx), int(v))][int(target)] += 1

n_ambiguous = sum(1 for c in key_targets.values() if len(c) > 1)
print(f"Ambiguous (prefix, context, validation) combos: "
      f"{n_ambiguous}/{len(key_targets)} ({n_ambiguous/len(key_targets)*100:.2f}%)")

majority = {k: c.most_common(1)[0][0] for k, c in key_targets.items()}
hit = sum(1 for seq, mask, ctx, v, t in zip(val_x, val_mask, val_ctx, val_val, val_y)
          if majority.get((tuple(seq[mask].tolist()), int(ctx), int(v))) == int(t))
print(f"Lookup-table accuracy on held-out validation: {hit/len(val_y)*100:.2f}%")

## 5. Model layers

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


class PositionalEmbedding(layers.Layer):
    """Learned positional embedding, added to token embeddings."""

    def __init__(self, max_sequence_length, embedding_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_sequence_length = max_sequence_length
        self.embedding_dim = embedding_dim
        self.supports_masking = True  # pass the incoming mask straight through

    def build(self, input_shape):
        self.position_embedding = self.add_weight(
            name="position_embedding",
            shape=(self.max_sequence_length, self.embedding_dim),
            initializer="uniform",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        sequence_length = tf.shape(inputs)[1]
        return inputs + self.position_embedding[:sequence_length]

    def compute_mask(self, inputs, mask=None):
        return mask

    def get_config(self):
        config = super().get_config()
        config.update({
            "max_sequence_length": self.max_sequence_length,
            "embedding_dim": self.embedding_dim,
        })
        return config


class ExpandAttentionMask(layers.Layer):
    """(batch, seq_len) boolean padding mask -> (batch, 1, seq_len) for MultiHeadAttention."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = False  # this layer *produces* the mask, it doesn't consume one

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, padding_mask):
        return tf.cast(padding_mask[:, tf.newaxis, :], tf.bool)

    def compute_mask(self, inputs, mask=None):
        # This layer's output is an attention-mask tensor, not a sequence with a
        # keras-managed mask. Stop propagation explicitly instead of staying silent.
        return None


class TransformerBlock(layers.Layer):
    def __init__(self, embedding_dim, num_heads, feed_forward_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.feed_forward_dim = feed_forward_dim
        self.dropout_rate = dropout_rate
        self.supports_masking = True

        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads,
            dropout=dropout_rate,
            name="self_attention",
        )
        self.feed_forward_network = tf.keras.Sequential(
            [
                layers.Dense(feed_forward_dim, activation="relu"),
                layers.Dense(embedding_dim),
                layers.Dropout(dropout_rate),
            ],
            name="feed_forward",
        )
        self.layer_norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.layer_norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.dropout_2 = layers.Dropout(dropout_rate)

    def build(self, input_shape):
        # Building sublayers explicitly here (rather than relying only on their
        # own lazy build-on-first-call) is what Keras 3 wants from a composite
        # layer that creates weight-owning sublayers in __init__.
        self.self_attention.build(input_shape, input_shape)
        self.feed_forward_network.build(input_shape)
        self.layer_norm_1.build(input_shape)
        self.layer_norm_2.build(input_shape)
        self.dropout_1.build(input_shape)
        self.dropout_2.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, attention_mask=None, training=None):
        attention_output = self.self_attention(
            query=inputs,
            key=inputs,
            value=inputs,
            attention_mask=attention_mask,
            training=training,
        )
        attention_output = self.dropout_1(attention_output, training=training)
        first_output = self.layer_norm_1(inputs + attention_output)

        feed_forward_output = self.feed_forward_network(first_output, training=training)
        feed_forward_output = self.dropout_2(feed_forward_output, training=training)

        return self.layer_norm_2(first_output + feed_forward_output)

    def compute_mask(self, inputs, mask=None):
        return mask

    def get_config(self):
        config = super().get_config()
        config.update({
            "embedding_dim": self.embedding_dim,
            "num_heads": self.num_heads,
            "feed_forward_dim": self.feed_forward_dim,
            "dropout_rate": self.dropout_rate,
        })
        return config


class GatherLastToken(layers.Layer):
    """
    Pools a (batch, seq_len, dim) sequence down to (batch, dim) by taking the
    LAST position.

    This is only correct because the sequences are LEFT-padded, so the final
    real token is always at index -1, for every row, regardless of how many
    real tokens precede it. It does NOT take an attention mask as input and
    does NOT compute sum(mask) - 1 -- that formula is the right-padding
    formula and silently points at a padding slot on left-padded data (see
    the validation assertion at data-load time, which checks mask[:, -1] is
    always True before this layer is ever used).
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # The output is a pooled, fixed-size vector -- there is no longer a
        # time axis, so there is nothing meaningful to mask downstream.
        self.supports_masking = False

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, inputs):
        return inputs[:, -1, :]

    def compute_mask(self, inputs, mask=None):
        return None

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[2])


def build_backbone(input_ids, padding_mask, vocab_size, embedding_dim, num_heads,
                    feed_forward_dim, num_blocks, dropout_rate, max_seq_len, prefix=""):
    """Shared token+position+transformer-blocks backbone used by every model
    variant below. Returns (sequence_output, token_embeddings).

    Masking design: the Embedding layer does NOT use mask_zero. Padding is
    handled by a single explicit source of truth -- the `padding_mask` input
    -- expanded once and threaded through every TransformerBlock's
    attention_mask argument. Keeping mask_zero=True on the Embedding *and*
    passing an explicit mask is redundant: it creates two separate masking
    mechanisms that the custom layers below (GatherLastToken in particular)
    were never designed to reconcile, which is exactly what produced the
    "custom layers do not implement mask propagation" warnings. Turning off
    the automatic mask keeps one unambiguous mechanism end to end.
    """
    token_embeddings = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=False,
        name=f"{prefix}token_embedding",
    )(input_ids)

    x = PositionalEmbedding(
        max_seq_len, embedding_dim, name=f"{prefix}positional_embedding",
    )(token_embeddings)

    expanded_mask = ExpandAttentionMask(
        name=f"{prefix}expanded_attention_mask",
    )(padding_mask)

    for block_number in range(num_blocks):
        x = TransformerBlock(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            feed_forward_dim=feed_forward_dim,
            dropout_rate=dropout_rate,
            name=f"{prefix}transformer_block_{block_number + 1}",
        )(x, attention_mask=expanded_mask)

    return x, token_embeddings

In [ ]:
"""
FieldSimilarityHead -- replaces the fixed Dense(NUM_CLASSES) output layer.

Standard softmax classification: logits = session_vector @ W + b, where W
is a (embedding_dim, num_classes) matrix of NUM_CLASSES independently
learned vectors, one per class, fixed at training time. No way to add a
class later without adding a row to W and retraining.

This layer instead learns a *shared* text encoder that turns any field's
one-sentence description into a vector. Logits = session_vector @
field_description_vectors.T. The known fields' vectors are computed from
their descriptions using that shared encoder -- so a brand-new field,
described in the same style, gets a real vector from the same encoder
without touching any trained weights.

This is the standard "dual encoder" / zero-shot classification pattern:
one encoder for the input, one (shared, generalizable) encoder for the
candidate labels, prediction = similarity.
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers


def build_description_vocab(descriptions):
    """Tiny whitespace tokenizer + vocab, no external dependencies."""
    vocab = {"<pad>": 0, "<unk>": 1}
    for desc in descriptions:
        for word in desc.lower().replace("-", " ").replace(",", " ").split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab


def encode_description(description, vocab, max_desc_len):
    words = description.lower().replace("-", " ").replace(",", " ").split()
    ids = [vocab.get(w, vocab["<unk>"]) for w in words][:max_desc_len]
    return ids + [vocab["<pad>"]] * (max_desc_len - len(ids))


class FieldSimilarityHead(layers.Layer):
    """
    Predicts by comparing a session vector against field-description
    vectors, instead of a fixed per-class weight matrix.

    - `field_names`, `field_descriptions`: the known fields at training
      time, in a fixed order (index i is class i).
    - `vocab`: word -> id, built once over the training descriptions and
      reused at inference (including for brand-new field descriptions).
    - The word-embedding + pooling + projection sublayers are the *shared,
      trainable* text encoder. They are what generalizes to new fields.
    """

    def __init__(self, field_names, field_descriptions, vocab, max_desc_len,
                 text_embed_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.field_names = list(field_names)
        self.field_descriptions = list(field_descriptions)
        self.vocab = vocab
        self.max_desc_len = max_desc_len
        self.text_embed_dim = text_embed_dim
        self.output_dim = output_dim

        encoded = [encode_description(d, vocab, max_desc_len) for d in field_descriptions]
        self.description_token_ids = tf.constant(encoded, dtype=tf.int32)  # (num_fields, max_desc_len)

        self.word_embedding = layers.Embedding(len(vocab), text_embed_dim, mask_zero=True,
                                                 name="field_description_word_embedding")
        self.text_projection = layers.Dense(output_dim, name="field_description_projection")
        self.session_projection = layers.Dense(output_dim, name="session_vector_projection")

    def build(self, input_shape):
        self.word_embedding.build((None, self.max_desc_len))
        self.text_projection.build((None, self.text_embed_dim))
        self.session_projection.build(input_shape)
        super().build(input_shape)

    def encode_descriptions(self, token_ids):
        """token_ids: (n, max_desc_len) -> (n, output_dim). Used both for the
        known fields (internally) and for scoring brand-new descriptions at
        inference (called directly, outside the Keras graph)."""
        word_vectors = self.word_embedding(token_ids)                 # (n, max_desc_len, text_embed_dim)
        mask = tf.cast(token_ids != 0, tf.float32)[:, :, tf.newaxis]  # ignore padding in the average
        pooled = tf.reduce_sum(word_vectors * mask, axis=1) / tf.maximum(tf.reduce_sum(mask, axis=1), 1.0)
        return self.text_projection(pooled)                           # (n, output_dim)

    def call(self, session_vector):
        field_vectors = self.encode_descriptions(self.description_token_ids)  # (num_fields, output_dim)
        session_vector = self.session_projection(session_vector)              # (batch, output_dim)
        logits = tf.matmul(session_vector, field_vectors, transpose_b=True)   # (batch, num_fields)
        return logits

    def score_new_description(self, session_vector_batch, description):
        """Score ONE new, never-trained-on field description against a batch
        of already-computed session vectors. This is the new capability:
        no retraining, just a forward pass through the same shared encoder."""
        token_ids = tf.constant([encode_description(description, self.vocab, self.max_desc_len)],
                                 dtype=tf.int32)
        new_field_vector = self.encode_descriptions(token_ids)                # (1, output_dim)
        session_vector = self.session_projection(session_vector_batch)        # (batch, output_dim)
        return tf.matmul(session_vector, new_field_vector, transpose_b=True)[:, 0]  # (batch,)

    def get_config(self):
        config = super().get_config()
        config.update({
            "field_names": self.field_names, "field_descriptions": self.field_descriptions,
            "vocab": self.vocab, "max_desc_len": self.max_desc_len,
            "text_embed_dim": self.text_embed_dim, "output_dim": self.output_dim,
        })
        return config

## 6. Field descriptions (editable catalog -- see section 9)

In [ ]:
FIELD_DESCRIPTIONS = {
    "full_name": "the customer's legal full name",
    "email_address": "the customer's email address for contact and notifications",
    "date_of_birth": "the customer's date of birth for age verification",
    "gender": "the customer's gender",
    "mobile_phone_number": "the customer's mobile phone number",
    "contact_phone_number": "a phone number to reach the customer",
    "home_address": "the customer's home address",
    "physical_address": "the customer's physical address for service delivery",
    "commercial_address": "the business customer's commercial premises address",
    "identification_document_type": "which type of government ID the customer will provide",
    "identification_document_number": "the ID document's official number",
    "identification_document": "a government-issued identification document",
    "selfie_liveness_check": "a live selfie photo to verify the customer is a real person",
    "face_match_verification": "matching the customer's selfie to their ID photo",
    "profession": "the customer's job or profession",
    "income": "the customer's stated income for financial profiling",
    "assets": "the customer's declared assets",
    "us_person_status": "whether the customer is a US person for tax reporting",
    "politically_exposed_person_status": "whether the customer holds a prominent public position",
    "terms_and_conditions_acceptance": "the customer's agreement to the terms and conditions",
    "beneficial_owner_information": "who ultimately owns or controls the business",
    "business_relationship_purpose": "why the business is opening this account",
    "business_relationship_nature": "the nature of the business's ongoing relationship with the provider",
    "ownership_and_control_information": "details of who owns and controls the business",
    "subscriber_type": "whether this is a new or existing telecom subscriber",
    "service_type": "which telecom service the customer is signing up for",
    "contract_type": "the length and type of contract the customer is signing up for",
    "payment_method": "how the customer will pay -- card, bank transfer, or direct debit",
    "paperless_billing_preference": "whether the customer wants paper or electronic billing",
    "phone_service_selection": "which phone service plan the customer selected",
    "internet_service_selection": "which internet service plan the customer selected",
    "streaming_addons_selection": "optional streaming add-ons the customer wants",
    "subscriber_number": "the new subscriber/account number assigned at activation",
}
descriptions_in_order = [FIELD_DESCRIPTIONS[f] for f in field_vocab]
assert len(descriptions_in_order) == NUM_CLASSES
vocab = build_description_vocab(descriptions_in_order)
MAX_DESC_LEN = 14
print("Description word vocab size:", len(vocab))

## 7. Build and train

In [ ]:
EMBEDDING_DIM, NUM_HEADS, FEED_FORWARD_DIM, NUM_BLOCKS, DROPOUT = 64, 4, 128, 2, 0.15
VOCAB_SIZE = NUM_CLASSES + 1

input_ids = layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
attention_mask = layers.Input(shape=(MAX_LEN,), dtype=tf.bool, name="attention_mask")
industry_id = layers.Input(shape=(), dtype=tf.int32, name="industry_id")
context_id = layers.Input(shape=(), dtype=tf.int32, name="context_id")
validation_id = layers.Input(shape=(), dtype=tf.int32, name="validation_id")

sequence_output, _ = build_backbone(
    input_ids, attention_mask, VOCAB_SIZE, EMBEDDING_DIM, NUM_HEADS,
    FEED_FORWARD_DIM, NUM_BLOCKS, DROPOUT, MAX_LEN,
)
pooled = GatherLastToken(name="last_valid_token")(sequence_output)

industry_embedding = layers.Embedding(len(industry_to_id), 8, name="industry_embedding")(industry_id)
context_embedding = layers.Embedding(len(context_to_id), 8, name="context_embedding")(context_id)
validation_embedding = layers.Embedding(len(validation_to_id), 4, name="validation_embedding")(validation_id)

combined = layers.Concatenate()([pooled, industry_embedding, context_embedding, validation_embedding])
session_vector = layers.Dense(128, activation="relu")(combined)
session_vector = layers.Dropout(DROPOUT)(session_vector)

field_head = FieldSimilarityHead(
    field_names=field_vocab, field_descriptions=descriptions_in_order,
    vocab=vocab, max_desc_len=MAX_DESC_LEN, text_embed_dim=32, output_dim=64,
    name="field_similarity_head",
)
logits = field_head(session_vector)
outputs = layers.Activation("softmax", name="next_field_probabilities")(logits)

model = tf.keras.Model(
    inputs=[input_ids, attention_mask, industry_id, context_id, validation_id],
    outputs=outputs, name="onboarding_transformer_flexible",
)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=2e-4),
    loss="sparse_categorical_crossentropy",
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top_3_accuracy")],
)
model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", mode="min", patience=3, restore_best_weights=True,
)
history = model.fit(
    [train_x, train_mask, train_ind, train_ctx, train_val], train_y,
    validation_data=([val_x, val_mask, val_ind, val_ctx, val_val], val_y),
    epochs=25, batch_size=512, verbose=1, callbacks=[early_stopping],
)
print("Epochs run:", len(history.history["loss"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Top-1 accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## Status: trained.

Next: Issue #21 (Evaluate Next-Field Prediction).